In [10]:
# Ex4 - abel problem
import numpy as np
import matplotlib.pyplot as plt

n = 1024

x = lambda s: np.where(
    (s > 0.3) & (s < 0.7),
    np.sin(np.pi * (s - 0.3) / 0.4),
    np.where((s > 0.75) & (s < 0.95), 0.5 * np.sin(np.pi * (s - 0.75) / 0.2),0.0))


s_grid = np.array([i/n for i in range(n)])
t_grid = s_grid + 0.5*1/n

x_grid = x(s_grid)

A = np.zeros((n,n))
for i,s in enumerate(s_grid):
    for j,t in enumerate(t_grid):
        if t-s >= 0:
            A[i,j] = 1 / np.sqrt(t-s) * 1/n     # 1/n comes from numerical integral, we have to consider the measure of our intervals!
        else:
            A[i,j] = 0

print(np.round(A,2))

# Perform SVD: A = U @ diag(s) @ Vt
U, s, Vt = np.linalg.svd(A)

# Identify singular values close to zero
tolerance = 1e-13
null_mask = s <= tolerance

# The rows of Vt corresponding to small singular values form the kernel basis
kernel_basis = Vt[null_mask].T
print("Kernel basis:\n", kernel_basis)


[[0.04 0.03 0.02 ... 0.   0.   0.  ]
 [0.   0.04 0.03 ... 0.   0.   0.  ]
 [0.   0.   0.04 ... 0.   0.   0.  ]
 ...
 [0.   0.   0.   ... 0.04 0.03 0.02]
 [0.   0.   0.   ... 0.   0.04 0.03]
 [0.   0.   0.   ... 0.   0.   0.04]]
Kernel basis:
 []


# Exercise 4

b)
add noise of 1% and 10%
c)
new function:
$$
x(s)=
\begin{cases}
\sin\!\left(\dfrac{\pi(s-0.3)}{0.4}\right),
& 0.3<s<0.7,\\[6pt]
\dfrac{1}{2}\sin\!\left(\dfrac{\pi(s-0.75)}{0.2}\right),
& 0.75<s<0.95,\\[6pt]
0, & \text{otherwise}.
\end{cases}
$$

In [ ]:
# Ex4 - abel problem
import numpy as np
import matplotlib.pyplot as plt

n = 10
# x = lambda s: s**2 * np.cos(np.pi * s / 2)

x = lambda s: np.where(
    (s > 0.3) & (s < 0.7),
    np.sin(np.pi * (s - 0.3) / 0.4),
    np.where((s > 0.75) & (s < 0.95), 0.5 * np.sin(np.pi * (s - 0.75) / 0.2),0.0))

s_grid = np.array([i/n for i in range(64)])
t_grid = s_grid + 0.5*1/n

x_grid = x(s_grid)

A = np.zeros((n,n))
for i,s in enumerate(s_grid):
    for j,t in enumerate(t_grid):
        if t-s >= 0:
            A[i,j] = 1 / np.sqrt(t-s) * 1/n     # 1/n comes from numerical integral, we have to consider the measure of our intervals!
        else:
            A[i,j] = 0

print(A)
y = A @ x_grid

maxIter = 10000
k_array = [1,2,3,6,15]
# Ex4b) add noise of 1% and 10%
d_array = np.array([0,0.01,0.1])

omega = 1.5 / np.linalg.norm(A,ord=2)**2
# LANDWEBER algorithm - classic

relativeError_LW = np.zeros((len(d_array),maxIter))
minError_ind_LW = np.full(len(d_array),maxIter)
minError_LW = np.full(len(d_array),np.linalg.norm(x_grid))      # initial value

x_delta = np.zeros(n)

for i,percent in enumerate(d_array):
    # noise
    np.random.seed(42)
    d = np.random.randn(n)
    d = d / np.linalg.norm(d) * percent * np.linalg.norm(y)
    y_delta = y + d

    #for k=0
    xk = np.zeros(n)

    
    for k in range(maxIter):
        xk = xk + omega * A.T @ (y_delta - A@xk)
        relativeError_LW[i,k] = np.linalg.norm( xk - x_grid ) / np.linalg.norm( x_grid)

        if relativeError_LW[i,k] < minError_LW[i]:
            minError_LW[i] = relativeError_LW[i,k]
            minError_ind_LW[i] = k
            x_delta = xk

        if k+1 in k_array: 
            plt.plot(s_grid,xk,label=f"x_{k}")

    plt.plot(s_grid,x_grid,label="initial x",color="black",linestyle="--")
    plt.plot(s_grid,x_delta,label="best recovered x",color="red",lw=2)
    plt.xlabel("s")
    plt.ylabel("x(s)")
    plt.legend()
    plt.title(f"LW solutions (delta={percent})")
    plt.grid(True)
    plt.ylim(-0.2,1)

    plt.show() 

# region PLOT
fig_LW, ax_LW = plt.subplots(figsize=(9, 5))
iterations = np.arange(1, maxIter + 1)

print("\nLandweber — best reconstruction for each noise level")
print(f"{'Noise':>8} {'Best iteration':>16} {'Min. rel. error':>18}")
print("-" * 46)

for i, percent in enumerate(d_array):
    best_idx = np.argmin(relativeError_LW[i])
    k_opt = best_idx + 1
    error_opt = relativeError_LW[i, best_idx]

    line, = ax_LW.loglog(iterations,relativeError_LW[i],lw=2,label=(f"Noise {percent:.0%}: "f"$k_{{opt}}={k_opt}$, min. error {error_opt:.2%}"))
    color = line.get_color()

    ax_LW.axvline(k_opt, color=color, linestyle="--", alpha=0.7)
    ax_LW.plot(k_opt, error_opt,marker="o", color=color, markersize=7,markeredgecolor="white", zorder=5)

    print(f"{percent:8.0%} {k_opt:16d} {error_opt:18.6f}")

ax_LW.set_xlabel("Iteration $k$")
ax_LW.set_ylabel(r"Relative error $\|x^{[k]}-x\|_2/\|x\|_2$")
ax_LW.set_title("LANDWEBER: relative reconstruction error")
ax_LW.grid(True, which="both", alpha=0.25)
ax_LW.legend(title="Dashed lines and dots mark the minima")
fig_LW.tight_layout()
plt.show()
# endregion


# region projectedLANDWEBER algorithm
relativeError_pLW = np.zeros((len(d_array),maxIter))
minError_ind_pLW = np.full(len(d_array),maxIter)
minError_pLW = np.full(len(d_array),np.linalg.norm(x_grid))      # initial value

x_delta = np.zeros(n)

P = lambda z: np.maximum(0, z)

for i,percent in enumerate(d_array):
    # noise
    np.random.seed(42)
    d = np.random.randn(n)
    d = d / np.linalg.norm(d) * percent * np.linalg.norm(y)
    y_delta = y + d

    xk = np.zeros(n)

    for k in range(maxIter):
        xk = P(xk + omega * A.T @ (y_delta - A@xk))
        relativeError_pLW[i,k] = np.linalg.norm( xk - x_grid ) / np.linalg.norm( x_grid)

        if relativeError_pLW[i,k] < minError_pLW[i]:
            minError_pLW[i] = relativeError_pLW[i,k]
            minError_ind_pLW[i] = k
            x_delta = xk

        if k+1 in k_array: 
            plt.plot(s_grid,xk,label=f"x_{k}")

    plt.plot(s_grid,x_grid,label="initial x",color="black",linestyle="--")
    plt.plot(s_grid,x_delta,label="best recovered x",color="red",lw=2)
    plt.xlabel("s")
    plt.ylabel("x(s)")
    plt.legend()
    plt.title(f"Projected LW solutions (delta={percent})")
    plt.grid(True)
    plt.ylim(-0.2,1)

    plt.show() 
# endregion

# region PLOT
fig_pLW, ax_pLW = plt.subplots(figsize=(9, 5))
iterations = np.arange(1, maxIter + 1)

print("\nprojected LW — best reconstruction for each noise level")
print(f"{'Noise':>8} {'Best iteration':>16} {'Min. rel. error':>18}")
print("-" * 46)

for i, percent in enumerate(d_array):
    best_idx = np.argmin(relativeError_pLW[i])
    k_opt = best_idx + 1
    error_opt = relativeError_pLW[i, best_idx]

    line, = ax_pLW.loglog(iterations,relativeError_pLW[i],lw=2,label=(f"Noise {percent:.0%}: "f"$k_{{opt}}={k_opt}$, min. error {error_opt:.2%}"))
    color = line.get_color()

    ax_pLW.axvline(k_opt, color=color, linestyle="--", alpha=0.7)
    ax_pLW.plot(k_opt, error_opt,marker="o", color=color, markersize=7,markeredgecolor="white", zorder=5)

    print(f"{percent:8.0%} {k_opt:16d} {error_opt:18.6f}")

ax_pLW.set_xlabel("Iteration $k$")
ax_pLW.set_ylabel(r"Relative error $\|x^{[k]}-x\|_2/\|x\|_2$")
ax_pLW.set_title("projected LW: relative reconstruction error")
ax_pLW.grid(True, which="both", alpha=0.25)
ax_pLW.legend(title="Dashed lines and dots mark the minima")
fig_pLW.tight_layout()
plt.show()


print("\nComparison: LW vs. projected LW")
print(
    f"{'Noise':>7} | {'LW':^25} | {'pLW':^25}"
)
print(
    f"{'':7} | {'k_opt':>8} {'Min. error':>16} | "
    f"{'k_opt':>8} {'Min. error':>16}"
)
print("-" * 65)

for i, percent in enumerate(d_array):
    idx_cgls = np.nanargmin(relativeError_LW[i])
    idx_lw = np.nanargmin(relativeError_pLW[i])

    print(
        f"{percent:7.0%} | "
        f"{idx_cgls + 1:8d} {relativeError_LW[i, idx_cgls]:16.2%} | "
        f"{idx_lw + 1:8d} {relativeError_pLW[i, idx_lw]:16.2%}"
    )
# endregion


NameError: name 's_grid' is not defined

wenn wir zu begin wissen, dass unsere funktion $\geq 0$ ist, wird das bei der projektion verwendet und nach jeder iteration die funktion wieder aus $\geq 0$ gesetzt